# Goodreads Book Genre Trends — Data Preparation

## Overview

This notebook prepares the Goodreads book dataset for exploratory data analysis and relational database construction.

The source data consists of numerous Parquet files organized by Goodreads genre or category. These files contain overlapping book records, meaning the same book may appear in multiple source files. The preparation process combines the source files, cleans and standardizes important fields, identifies unique books, normalizes authors and genres, and constructs the relationships needed for a relational database.

The final prepared datasets are saved as individual Parquet files and will be used to populate the SQLite database used throughout the capstone project.

---

## Objectives

This notebook is designed to:

1. Inspect and evaluate the source Parquet datasets.
2. Combine the individual genre files into a single dataset.
3. Select and retain the fields necessary for analysis and database construction.
4. Clean publication-year and categorical data.
5. Normalize author information.
6. Create a consistent unique identifier for each book.
7. Filter the dataset to the project's publication-year range of **1900–2016**.
8. Normalize Goodreads genre information.
9. Identify and preserve relationships between books, authors, genres, and source datasets.
10. Construct the DataFrames corresponding to the relational database tables.
11. Assign primary-key and foreign-key identifiers.
12. Validate the prepared relational data.
13. Save the finalized datasets for SQLite database creation.

---

## Source Data

The source dataset contains Goodreads book information distributed across multiple Parquet files in:

`Data/Goodreads_Books/genres_top100`

Each Parquet file represents a Goodreads genre or category and may contain books that also appear in other source files.

The primary fields used in this project include:

- `id` — source dataset identifier
- `name` — book title
- `author` — author or authors associated with the book
- `genres` — Goodreads genre classifications
- `pub_year` — publication year
- `star_rating` — average Goodreads rating
- `num_ratings` — number of Goodreads ratings
- `isbn_clean` — cleaned ISBN value

The original Parquet filename is also retained as `source_genre` to identify which source dataset contained each book.

---

## Data Preparation Strategy

Because the source files contain substantial overlap, the data is not simply concatenated and treated as a flat dataset.

Instead, the preparation process separates the information into logical entities and relationships.

The resulting relational structure consists of seven datasets:

### Entity Tables

- **BOOKS** — one record per unique book.
- **AUTHORS** — one record per unique author.
- **GENRES** — one record per unique Goodreads genre.
- **SOURCE_GENRES** — one record per original source dataset.

### Relationship Tables

- **BOOK_AUTHORS** — connects books and authors.
- **BOOK_GENRES** — connects books and Goodreads genres.
- **BOOK_SOURCE_GENRES** — connects books to the original source datasets in which they appeared.

This structure allows a single book to have multiple authors, multiple genres, and multiple source classifications without unnecessarily duplicating the book's core information.

---

## Book Identification

A `book_key` is created to identify unique books across the overlapping source files.

When an ISBN is available, the ISBN is used as the primary component of the key.

When an ISBN is unavailable, a fallback key is constructed using:

- Book title
- Author(s)
- Publication year

This approach allows duplicate records across different source files to be identified while retaining books for which an ISBN is unavailable.

---

## Publication-Year Filtering

For the purposes of this analysis, books are restricted to publication years from **1900 through 2016**, inclusive.

This provides a consistent historical period for examining changes in book genres and publication trends over time.

---

## Genre Normalization

The Goodreads `genres` field may contain multiple genre classifications for a single book.

The genre information is therefore transformed into individual book/genre relationships.

Broad labels such as:

- `fiction`
- `non-fiction`
- `nonfiction`

are excluded from the specific genre analysis because they provide less useful information than the more detailed genre classifications.

---

## Relational Database Preparation

The prepared DataFrames correspond to the relational database design that will later be implemented in SQLite.

The primary keys are represented by:

- `book_id`
- `author_id`
- `genre_id`
- `source_genre_id`

The relationship tables use these values as foreign keys.

Before saving the prepared datasets, the notebook validates that:

- Primary keys are unique.
- Foreign keys successfully map to their corresponding entities.
- Relationship records do not reference nonexistent books.
- Duplicate relationships have been removed.

---

## Final Output

The finalized relational datasets are saved in:

`Data/prepared`

The output files are:

- `books.parquet`
- `authors.parquet`
- `book_authors.parquet`
- `genres.parquet`
- `book_genres.parquet`
- `source_genres.parquet`
- `book_source_genres.parquet`

These files provide the clean, structured input needed to build and populate the SQLite database used for the capstone project.

---

## Next Step

The next stage of the project is to use these prepared datasets to **create and populate the SQLite relational database**, enforce primary-key and foreign-key relationships, and perform SQL queries for exploratory and analytical purposes.

## 1. Inspect Source Dataset Files

Before loading the Goodreads data, this step identifies the available Parquet files and calculates their individual and combined file sizes.

This provides an overview of the size of the source dataset and helps establish the scale of the data that will be processed throughout the preparation workflow.

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import os
import pandas as pd
import numpy as np


total = 0
folder = Path("../Data/Goodreads_Books/genres_top100")
for f in folder.glob("*.parquet"):
    size = os.path.getsize(f)
    total += size
    print(f"{f.name}: {size / 1e6:.1f} MB")
print(f"Total: {total / 1e6:.1f} MB")

action.parquet: 4.6 MB
adult.parquet: 19.4 MB
adventure.parquet: 13.6 MB
amazon.parquet: 6.0 MB
american_history.parquet: 5.1 MB
animals.parquet: 9.1 MB
anthologies.parquet: 6.8 MB
art.parquet: 15.2 MB
audiobook.parquet: 39.5 MB
bdsm.parquet: 5.3 MB
biography.parquet: 25.0 MB
biography_memoir.parquet: 6.7 MB
book_club.parquet: 7.6 MB
british_literature.parquet: 7.1 MB
business.parquet: 9.7 MB
chick_lit.parquet: 8.4 MB
childrens.parquet: 33.9 MB
christian.parquet: 13.5 MB
christianity.parquet: 5.7 MB
christian_fiction.parquet: 5.2 MB
christmas.parquet: 4.6 MB
classics.parquet: 15.9 MB
comics.parquet: 25.8 MB
comic_book.parquet: 4.5 MB
contemporary.parquet: 41.4 MB
contemporary_romance.parquet: 20.5 MB
cookbooks.parquet: 9.5 MB
cooking.parquet: 5.8 MB
crime.parquet: 16.3 MB
drama.parquet: 5.2 MB
ebooks.parquet: 22.0 MB
economics.parquet: 5.1 MB
education.parquet: 5.6 MB
erotica.parquet: 14.2 MB
essays.parquet: 5.2 MB
family.parquet: 6.2 MB
fantasy.parquet: 56.3 MB
fiction.parquet: 122.9 

## 2. Inspect Available Dataset Columns

A sample Parquet file is loaded to inspect the columns available in the source dataset.

This helps determine which fields are relevant to the analysis and which can be excluded before combining the files.

In [2]:
folder = Path("../Data/Goodreads_Books/genres_top100")

# First: check what columns exist and drop anything you don't need
sample = pq.read_table(next(folder.glob("*.parquet")))
print(sample.column_names)

['id', 'name', 'author', 'url', 'genres', 'summary_clean', 'pub_year', 'star_rating', 'num_ratings', 'isbn_clean']


## 3. Combine the Goodreads Genre Datasets

The required columns are selected from each Parquet file and combined into a single dataset.

The original filename is also stored in a new `genre` column. This preserves the source genre/category associated with each record and allows the source classification to be analyzed later.

The resulting tables are concatenated into one large PyArrow table and saved as `all_genres_combined.parquet` for use in the remaining data-preparation steps.

In [4]:
folder = Path("../Data/Goodreads_Books/genres_top100")

keep_cols = ["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("genre", genre_col)
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
pq.write_table(combined_table, "../Data/Goodreads_Books/all_genres_combined.parquet")

## 4. Preview the Source Data

A sample of the selected columns is converted to a Pandas DataFrame and displayed to verify that the expected fields and values were loaded correctly before continuing with the data-preparation process.

In [5]:
sample = pq.read_table(next(folder.glob("*.parquet")), columns=["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"])
sample.to_pandas().head(10)

,id,name,author,genres,pub_year,star_rating,num_ratings,isbn_clean
0,615225.Sharpe_s_Devil,Sharpe's Devil,[Bernard Cornwell],"[Historical Fiction, Fiction, Historical, War,...",1992,4.14,8141,9780060932299
1,278794.Blood_Fever,Blood Fever,[Charlie Higson],"[Young Adult, Fiction, Adventure, Mystery, Thr...",2006,4.02,7516,9780786836628
2,23124285-detektiv-conan-vs-kaito-kid,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],"[Manga, Mystery, Komik, Indonesian Literature,...",2004,4.33,231,9783770476374
3,29280242-marvel-s-captain-america,Marvel's Captain America: Sub Rosa,[David McDonald],"[Marvel, Young Adult, Action, Superheroes]",2016,3.39,74,9781772752014
4,2844643-the-awakening,The Awakening,[Jerry Ahern],"[Post Apocalyptic, Action, Dystopia, Fiction, ...",1984,3.87,305,9780821714782
5,266485.Last_of_the_Breed,Last of the Breed,[Louis L'Amour],"[Fiction, Westerns, Adventure, Historical Fict...",1986,4.28,15161,NaN
6,23906444-terrible-tuesday,Terrible Tuesday,[Don Pendleton],"[Thriller, Fiction, Adventure, Action]",1979,4.02,292,9781497687622
7,3986318-terminal-freeze,Terminal Freeze,[Lincoln Child],"[Thriller, Fiction, Mystery, Horror, Science F...",2008,3.83,19403,9780385515511
8,377612.Eureka_Seven,"Eureka Seven: Psalms of Planets, Vol. 2",[Jinsei Kataoka],"[Manga, Science Fiction, Graphic Novels, Shone...",2005,4.05,268,9781594096914
9,14431469-devil-s-pass,Devil's Pass,[Sigmund Brouwer],"[Adventure, Young Adult, Canada, Fiction, Myst...",2012,3.83,595,9781554699384


## 5. Load and Combine the Source Data for Preparation

The selected columns from all Goodreads Parquet files are loaded and combined into a single Pandas DataFrame.

The original Parquet filename is stored as `source_genre` to preserve the source category for each record.

The `pub_year` column is converted to a numeric integer type, with invalid values converted to missing values. The `source_genre` column is converted to a categorical data type to reduce memory usage while working with the large combined dataset.

In [6]:
folder = Path("../Data/Goodreads_Books/genres_top100")
keep_cols = ["id", "name", "author", "genres", "pub_year", "star_rating", "num_ratings", "isbn_clean"]

tables = []
for file in folder.glob("*.parquet"):
    table = pq.read_table(file, columns=keep_cols)
    genre_col = pa.array([file.stem] * table.num_rows)
    table = table.append_column("source_genre", genre_col)  # from filename, as backup/comparison
    tables.append(table)

combined_table = pa.concat_tables(tables, promote_options="default")
df = combined_table.to_pandas()

# dtype cleanup — helps memory a lot with this many rows
df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce", downcast="integer")
df["source_genre"] = df["source_genre"].astype("category")

## 6. Inspect the Combined Dataset

The combined DataFrame is inspected to determine its dimensions, memory usage, column data types, and overall structure.

Because the dataset contains millions of records, monitoring memory usage is particularly important during the remaining data-preparation steps.

In [7]:
print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")
df.info()

(4820937, 9)
1700.75812 MB
<class 'pandas.DataFrame'>
RangeIndex: 4820937 entries, 0 to 4820936
Data columns (total 9 columns):
 #   Column        Dtype   
---  ------        -----   
 0   id            str     
 1   name          str     
 2   author        object  
 3   genres        object  
 4   pub_year      int16   
 5   star_rating   float64 
 6   num_ratings   int64   
 7   isbn_clean    str     
 8   source_genre  category
dtypes: category(1), float64(1), int16(1), int64(1), object(2), str(3)
memory usage: 592.1+ MB


## 7. Normalize Goodreads Genre Data

The Goodreads `genres` field contains multiple genre labels for each book. The genre data is transformed into a standardized list format and then exploded so that each book/genre combination occupies its own row.

Blank and missing genre values are removed, and broad labels such as `fiction`, `non-fiction`, and `nonfiction` are excluded because the analysis focuses on more specific genre classifications.

The resulting data is used to create:

- `book_genres` — the many-to-many relationship between books and genres.
- `genres` — a unique list of Goodreads genres.
- `year_genre_counts` — aggregated book/genre counts by publication year for later exploratory data analysis.

This step separates genre information from the book records and prepares it for the relational database structure.

In [31]:
# ---------------------------------------------------------
# Process Goodreads Genres
# ---------------------------------------------------------

# Convert the genres column into a clean list of genre names
df["genres_list"] = df["genres"].apply(
    lambda g: [str(x).strip().lower() for x in g]
    if isinstance(g, (list, tuple, np.ndarray))
    else []
)

# Create one row for each book/genre combination
exploded = df.explode("genres_list")

# Remove missing or blank genre values
exploded = exploded[
    exploded["genres_list"].notna() &
    (exploded["genres_list"] != "")
].copy()

# Remove the generic Fiction/Nonfiction labels
# These aren't useful for the more specific genre analysis.
fiction_labels = {
    "fiction",
    "non-fiction",
    "nonfiction"
}

exploded = exploded[
    ~exploded["genres_list"].isin(fiction_labels)
].copy()

# ---------------------------------------------------------
# Create book/genre relationship data
# ---------------------------------------------------------

# Keep the information needed to establish the relationship
# between a unique book and its genres.
book_genres = (
    exploded[
        ["book_key", "genres_list"]
    ]
    .drop_duplicates()
    .rename(columns={
        "genres_list": "genre_name"
    })
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Create a unique genre table
# ---------------------------------------------------------

genres = (
    book_genres[["genre_name"]]
    .drop_duplicates()
    .sort_values("genre_name")
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Create genre counts for EDA
# ---------------------------------------------------------

year_genre_counts = (
    exploded
    .groupby(["pub_year", "genres_list"])
    .size()
    .reset_index(name="count")
)

# ---------------------------------------------------------
# Inspect results
# ---------------------------------------------------------

print("Unique genres:", len(genres))
print("Book/genre relationships:", len(book_genres))

print("\nGenres:")
display(genres.head(20))

print("\nBook/Genre relationships:")
display(book_genres.head(10))

print("\nYear/Genre counts:")
display(year_genre_counts.head(10))

Unique genres: 1440
Book/genre relationships: 5532852

Genres:


,genre_name
0,10th century
1,11th century
2,12th century
3,13th century
4,14th century
5,15th century
6,16th century
7,17th century
8,1864 shenandoah campaign
9,18th century



Book/Genre relationships:


,book_key,genre_name
0,isbn:9780060932299,historical fiction
1,isbn:9780060932299,historical
2,isbn:9780060932299,war
3,isbn:9780060932299,adventure
4,isbn:9780060932299,military fiction
5,isbn:9780060932299,audiobook
6,isbn:9780060932299,action
7,isbn:9780060932299,ebooks
8,isbn:9780060932299,19th century
9,isbn:9780786836628,young adult



Year/Genre counts:


,pub_year,genres_list,count
0,1000,ancient,5
1,1000,anglo saxon,5
2,1000,anthropology,3
3,1000,asia,12
4,1000,asian literature,6
5,1000,australia,1
6,1000,biography,5
7,1000,business,14
8,1000,christian,9
9,1000,christian fiction,5


## 8. Filter Publication Years

The dataset is restricted to books with publication years between **1900 and 2016**, inclusive.

This range was selected to provide a consistent historical period for analyzing changes in book genres over time. Records with publication years outside this range are excluded from the prepared dataset.

The number of records before and after filtering is displayed to document the effect of this cleaning step.

In [ ]:
# ---------------------------------------------------------
# Filter publication years
# ---------------------------------------------------------

valid = df[
    (df["pub_year"] >= 1900) &
    (df["pub_year"] <= 2016)
].copy()

print("Total combined records:", len(df))
print("Records after year filter:", len(valid))

## 9. Clean and Normalize Author Information

The source `author` field may contain one or more authors and can include inconsistent whitespace or formatting.

A cleaning function is applied to standardize author names by removing unnecessary whitespace and excluding blank values. The cleaned results are stored in `authors_list`.

Keeping authors as a list at this stage allows books with multiple authors to be properly represented later through the `authors` and `book_authors` relational tables.

In [13]:
def clean_authors(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        return [
            " ".join(str(author).split()).strip()
            for author in value
            if str(author).strip()
        ]
    return []


valid["authors_list"] = valid["author"].apply(clean_authors)

# Inspect the results
display(df[["author", "authors_list"]].head(10))

,author,authors_list
0,[Bernard Cornwell],[Bernard Cornwell]
1,[Charlie Higson],[Charlie Higson]
2,[Gosho Aoyama],[Gosho Aoyama]
3,[David McDonald],[David McDonald]
4,[Jerry Ahern],[Jerry Ahern]
5,[Louis L'Amour],[Louis L'Amour]
6,[Don Pendleton],[Don Pendleton]
7,[Lincoln Child],[Lincoln Child]
8,[Jinsei Kataoka],[Jinsei Kataoka]
9,[Sigmund Brouwer],[Sigmund Brouwer]


## 10. Create a Unique Book Key

A consistent `book_key` is created to identify the same book across multiple source genre files.

When an ISBN is available, it is used as the primary component of the key because it provides a strong identifier for a book edition. When an ISBN is missing, a fallback key is created using the book title, author information, and publication year.

This key allows duplicate appearances of the same book to be identified while preserving books that do not have an ISBN in the source data.

In [14]:
# ---------------------------------------------------------
# Create a unique key for each book
# ---------------------------------------------------------

def create_book_key(row):
    # Use ISBN when available
    if pd.notna(row["isbn_clean"]) and str(row["isbn_clean"]).strip():
        return f"isbn:{str(row['isbn_clean']).strip()}"
    
    # Otherwise use title + authors + publication year
    authors = "|".join(sorted(row["authors_list"]))
    
    return (
        f"title:{str(row['name']).strip().lower()}|"
        f"author:{authors.lower()}|"
        f"year:{row['pub_year']}"
    )


valid["book_key"] = valid.apply(create_book_key, axis=1)

display(
    df[
        [
            "name",
            "authors_list",
            "pub_year",
            "isbn_clean",
            "book_key"
        ]
    ].head(10)
)

,name,authors_list,pub_year,isbn_clean,book_key
0,Sharpe's Devil,[Bernard Cornwell],1992,9780060932299,isbn:9780060932299
1,Blood Fever,[Charlie Higson],2006,9780786836628,isbn:9780786836628
2,Detektiv Conan vs. Kaito Kid,[Gosho Aoyama],2004,9783770476374,isbn:9783770476374
3,Marvel's Captain America: Sub Rosa,[David McDonald],2016,9781772752014,isbn:9781772752014
4,The Awakening,[Jerry Ahern],1984,9780821714782,isbn:9780821714782
5,Last of the Breed,[Louis L'Amour],1986,NaN,title:last of the breed|author:louis l'amour|y...
6,Terrible Tuesday,[Don Pendleton],1979,9781497687622,isbn:9781497687622
7,Terminal Freeze,[Lincoln Child],2008,9780385515511,isbn:9780385515511
8,"Eureka Seven: Psalms of Planets, Vol. 2",[Jinsei Kataoka],2005,9781594096914,isbn:9781594096914
9,Devil's Pass,[Sigmund Brouwer],2012,9781554699384,isbn:9781554699384


## 11. Validate Book Uniqueness

The newly created `book_key` is evaluated to determine how many unique books are represented in the filtered dataset.

The total number of valid records, unique book keys, and duplicate records are displayed. Duplicate records are expected because the same book may appear in multiple source genre files.

These duplicate records will be resolved through the relational database structure rather than simply discarded, allowing the relationships between books and genres to be preserved.

In [20]:
print("Total valid records:", len(valid))
print("Unique book keys:", valid["book_key"].nunique())
print("Duplicate records:", valid["book_key"].duplicated().sum())

Total valid records: 4352106
Unique book keys: 1487805
Duplicate records: 2864301


## 12. Create Relational Database DataFrames

The cleaned Goodreads data is transformed into separate DataFrames that correspond to the tables in the relational database.

The database is designed to reduce duplication and accurately represent relationships between books, authors, genres, and the original source datasets.

The following tables are created:

- **BOOKS** — one record for each unique book.
- **AUTHORS** — one record for each unique author.
- **BOOK_AUTHORS** — connects books to their authors through a many-to-many relationship.
- **GENRES** — one record for each unique Goodreads genre.
- **BOOK_GENRES** — connects books to their Goodreads genres through a many-to-many relationship.
- **SOURCE_GENRES** — contains the original genre/category represented by each source Parquet file.
- **BOOK_SOURCE_GENRES** — connects books to the source datasets in which they appeared.

This structure prepares the cleaned data for insertion into the SQLite relational database and preserves relationships that would be lost if the source records were simply deduplicated into a single flat table.

In [21]:
# =========================================================
# Create Relational Database DataFrames
# =========================================================

# ---------------------------------------------------------
# 1. BOOKS
# ---------------------------------------------------------
# One row per unique book.
#
# book_key was created during data preparation and is used
# to identify duplicate copies of the same book across the
# different Goodreads source files.
# ---------------------------------------------------------

books = (
    valid[
        [
            "book_key",
            "name",
            "pub_year",
            "star_rating",
            "num_ratings",
            "isbn_clean"
        ]
    ]
    .drop_duplicates(subset="book_key")
    .reset_index(drop=True)
)

print("BOOKS:", len(books))


# ---------------------------------------------------------
# 2. AUTHORS
# ---------------------------------------------------------
# Create one row per unique author.
#
# authors_list contains the cleaned list of authors for each
# book, so we explode it before creating the AUTHORS table.
# ---------------------------------------------------------

authors = (
    valid[["authors_list"]]
    .explode("authors_list")
    .rename(columns={"authors_list": "author_name"})
)

# Remove missing/blank authors
authors = authors[
    authors["author_name"].notna() &
    (authors["author_name"] != "")
].copy()

# Get each author only once
authors = (
    authors[["author_name"]]
    .drop_duplicates()
    .sort_values("author_name")
    .reset_index(drop=True)
)

print("AUTHORS:", len(authors))


# ---------------------------------------------------------
# 3. BOOK_AUTHORS
# ---------------------------------------------------------
# Establish the many-to-many relationship between books
# and authors.
# ---------------------------------------------------------

book_authors = (
    valid[
        [
            "book_key",
            "authors_list"
        ]
    ]
    .explode("authors_list")
    .rename(columns={"authors_list": "author_name"})
)

book_authors = book_authors[
    book_authors["author_name"].notna() &
    (book_authors["author_name"] != "")
].copy()

book_authors = (
    book_authors[
        [
            "book_key",
            "author_name"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("BOOK_AUTHORS relationships:", len(book_authors))


# ---------------------------------------------------------
# 4. GENRES
# ---------------------------------------------------------
# Process the Goodreads genre lists.
# ---------------------------------------------------------

valid["genres_list"] = valid["genres"].apply(
    lambda g: [
        str(x).strip().lower()
        for x in g
    ]
    if isinstance(g, (list, tuple, np.ndarray))
    else []
)

# Explode so each book/genre combination becomes a row
exploded = valid.explode("genres_list")

# Remove missing/blank genres
exploded = exploded[
    exploded["genres_list"].notna() &
    (exploded["genres_list"] != "")
].copy()

# Remove broad labels that aren't useful for our
# specific genre analysis
fiction_labels = {
    "fiction",
    "non-fiction",
    "nonfiction"
}

exploded = exploded[
    ~exploded["genres_list"].isin(fiction_labels)
].copy()


genres = (
    exploded[["genres_list"]]
    .rename(columns={"genres_list": "genre_name"})
    .drop_duplicates()
    .sort_values("genre_name")
    .reset_index(drop=True)
)

print("GENRES:", len(genres))


# ---------------------------------------------------------
# 5. BOOK_GENRES
# ---------------------------------------------------------
# Establish the many-to-many relationship between books
# and Goodreads genres.
# ---------------------------------------------------------

book_genres = (
    exploded[
        [
            "book_key",
            "genres_list"
        ]
    ]
    .rename(columns={"genres_list": "genre_name"})
    .drop_duplicates()
    .reset_index(drop=True)
)

print("BOOK_GENRES relationships:", len(book_genres))


# ---------------------------------------------------------
# 6. SOURCE_GENRES
# ---------------------------------------------------------
# These are the original Parquet filenames.
#
# Example:
# fantasy.parquet → fantasy
# science_fiction.parquet → science_fiction
# ---------------------------------------------------------

source_genres = (
    valid[["source_genre"]]
    .drop_duplicates()
    .sort_values("source_genre")
    .reset_index(drop=True)
)

print("SOURCE_GENRES:", len(source_genres))


# ---------------------------------------------------------
# 7. BOOK_SOURCE_GENRES
# ---------------------------------------------------------
# Establish which original source dataset(s) contained
# each book.
# ---------------------------------------------------------

book_source_genres = (
    valid[
        [
            "book_key",
            "source_genre"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "BOOK_SOURCE_GENRES relationships:",
    len(book_source_genres)
)

BOOKS: 1487805
AUTHORS: 482461
BOOK_AUTHORS relationships: 1488070
GENRES: 1433
BOOK_GENRES relationships: 4981671
SOURCE_GENRES: 100
BOOK_SOURCE_GENRES relationships: 4291574


## 13. Validate Relational DataFrames

A summary of the prepared relational DataFrames is displayed to verify the number of records and relationships that will be transferred into the SQLite database.

This provides a final checkpoint before database creation and helps confirm that each table contains data and that the many-to-many relationship tables have been generated successfully.

In [22]:
# =========================================================
# Validate Relational DataFrames
# =========================================================

print("\n========== DATABASE DATASET SUMMARY ==========")

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")


========== DATABASE DATASET SUMMARY ==========
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574


## 14. Validate Book Relationships

The relationship DataFrames are checked to ensure that every `book_key` in the `BOOK_AUTHORS`, `BOOK_GENRES`, and `BOOK_SOURCE_GENRES` tables corresponds to a valid book in the `BOOKS` table.

This verifies referential integrity before the data is inserted into SQLite and helps prevent orphaned relationship records.

In [23]:
# =========================================================
# Check that all relationships point to valid books
# =========================================================

missing_book_authors = ~book_authors["book_key"].isin(
    books["book_key"]
)

missing_book_genres = ~book_genres["book_key"].isin(
    books["book_key"]
)

missing_book_sources = ~book_source_genres["book_key"].isin(
    books["book_key"]
)

print(
    "Book-author records with missing books:",
    missing_book_authors.sum()
)

print(
    "Book-genre records with missing books:",
    missing_book_genres.sum()
)

print(
    "Book-source records with missing books:",
    missing_book_sources.sum()
)

Book-author records with missing books: 0
Book-genre records with missing books: 0
Book-source records with missing books: 0


## 15. Review Prepared Relational Data

The first records from each prepared DataFrame are displayed for a final visual inspection.

Reviewing the tables confirms that the data has been structured as expected before it is written to the SQLite database, including the book records, author and genre entities, and their associated relationship tables.

In [24]:
print("\nBOOKS")
display(books.head())

print("\nAUTHORS")
display(authors.head(10))

print("\nBOOK_AUTHORS")
display(book_authors.head(10))

print("\nGENRES")
display(genres.head(20))

print("\nBOOK_GENRES")
display(book_genres.head(10))

print("\nSOURCE_GENRES")
display(source_genres.head(20))

print("\nBOOK_SOURCE_GENRES")
display(book_source_genres.head(10))


BOOKS


,book_key,name,pub_year,star_rating,num_ratings,isbn_clean
0,isbn:9780060932299,Sharpe's Devil,1992,4.14,8141,9780060932299
1,isbn:9780786836628,Blood Fever,2006,4.02,7516,9780786836628
2,isbn:9783770476374,Detektiv Conan vs. Kaito Kid,2004,4.33,231,9783770476374
3,isbn:9781772752014,Marvel's Captain America: Sub Rosa,2016,3.39,74,9781772752014
4,isbn:9780821714782,The Awakening,1984,3.87,305,9780821714782



AUTHORS


,author_name
0,!
1,"""Albert"""
2,"""Big"" John McCarthy"
3,"""J"""
4,"""Janosch"""
5,"""Laughing"" Larry Berger"
6,"""Lebanon"" Levi Stoltzfus"
7,"""Markku Oksanen, Veikko Launis & Seppo Sajama"
8,"""Miss Naomi"""
9,"""Panda"" Andy McAllister"



BOOK_AUTHORS


...

,book_key,genre_name
0,isbn:9780060932299,historical fiction
1,isbn:9780060932299,historical
2,isbn:9780060932299,war
3,isbn:9780060932299,adventure
4,isbn:9780060932299,military fiction
5,isbn:9780060932299,audiobook
6,isbn:9780060932299,action
7,isbn:9780060932299,ebooks
8,isbn:9780060932299,19th century
9,isbn:9780786836628,young adult



SOURCE_GENRES


,source_genre
0,action
1,adult
2,adventure
3,amazon
4,american_history
5,animals
6,anthologies
7,art
8,audiobook
9,bdsm



BOOK_SOURCE_GENRES


,book_key,source_genre
0,isbn:9780060932299,action
1,isbn:9780786836628,action
2,isbn:9783770476374,action
3,isbn:9781772752014,action
4,isbn:9780821714782,action
5,title:last of the breed|author:louis l'amour|y...,action
6,isbn:9781497687622,action
7,isbn:9780385515511,action
8,isbn:9781594096914,action
9,isbn:9781554699384,action


## 16. Assign Database IDs

Unique integer IDs are assigned to the primary entity tables: `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES`.

Lookup dictionaries are then created to map each natural key or name to its corresponding database ID. These mappings will be used to replace text-based relationship values with integer foreign keys when preparing the relationship tables for SQLite.

Using numeric primary and foreign keys improves database structure and allows the relationship tables to reference the appropriate records efficiently.

In [26]:
# =========================================================
# Assign Database IDs
# =========================================================

# ---------------------------------------------------------
# BOOK IDs
# ---------------------------------------------------------

books = books.reset_index(drop=True)
books.insert(0, "book_id", books.index + 1)


# ---------------------------------------------------------
# AUTHOR IDs
# ---------------------------------------------------------

authors = authors.reset_index(drop=True)
authors.insert(0, "author_id", authors.index + 1)


# ---------------------------------------------------------
# GENRE IDs
# ---------------------------------------------------------

genres = genres.reset_index(drop=True)
genres.insert(0, "genre_id", genres.index + 1)


# ---------------------------------------------------------
# SOURCE GENRE IDs
# ---------------------------------------------------------

source_genres = source_genres.reset_index(drop=True)
source_genres.insert(0, "source_genre_id", source_genres.index + 1)


# ---------------------------------------------------------
# Create lookup dictionaries
# ---------------------------------------------------------

book_id_lookup = dict(
    zip(
        books["book_key"],
        books["book_id"]
    )
)

author_id_lookup = dict(
    zip(
        authors["author_name"],
        authors["author_id"]
    )
)

genre_id_lookup = dict(
    zip(
        genres["genre_name"],
        genres["genre_id"]
    )
)

source_genre_id_lookup = dict(
    zip(
        source_genres["source_genre"],
        source_genres["source_genre_id"]
    )
)


print("Book IDs:", len(book_id_lookup))
print("Author IDs:", len(author_id_lookup))
print("Genre IDs:", len(genre_id_lookup))
print("Source Genre IDs:", len(source_genre_id_lookup))

Book IDs: 1487805
Author IDs: 482461
Genre IDs: 1433
Source Genre IDs: 100


## 17. Convert Relationship Tables to Foreign Keys

The relationship tables are converted from text-based identifiers to numeric foreign keys using the lookup dictionaries created in the previous step.

Each relationship table is reduced to the foreign keys required to connect its records:

- `BOOK_AUTHORS` connects `book_id` to `author_id`.
- `BOOK_GENRES` connects `book_id` to `genre_id`.
- `BOOK_SOURCE_GENRES` connects `book_id` to `source_genre_id`.

Duplicate relationships are removed to ensure that each book-to-entity relationship is represented only once. The resulting relationship counts are displayed as a final validation check.

In [27]:
# =========================================================
# Convert Relationship Tables to Foreign Keys
# =========================================================

# ---------------------------------------------------------
# BOOK_AUTHORS
# ---------------------------------------------------------

book_authors["book_id"] = book_authors["book_key"].map(
    book_id_lookup
)

book_authors["author_id"] = book_authors["author_name"].map(
    author_id_lookup
)

book_authors = book_authors[
    ["book_id", "author_id"]
].drop_duplicates()


# ---------------------------------------------------------
# BOOK_GENRES
# ---------------------------------------------------------

book_genres["book_id"] = book_genres["book_key"].map(
    book_id_lookup
)

book_genres["genre_id"] = book_genres["genre_name"].map(
    genre_id_lookup
)

book_genres = book_genres[
    ["book_id", "genre_id"]
].drop_duplicates()


# ---------------------------------------------------------
# BOOK_SOURCE_GENRES
# ---------------------------------------------------------

book_source_genres["book_id"] = book_source_genres["book_key"].map(
    book_id_lookup
)

book_source_genres["source_genre_id"] = (
    book_source_genres["source_genre"].map(
        source_genre_id_lookup
    )
)

book_source_genres = book_source_genres[
    ["book_id", "source_genre_id"]
].drop_duplicates()


print("BOOK_AUTHORS:", len(book_authors))
print("BOOK_GENRES:", len(book_genres))
print("BOOK_SOURCE_GENRES:", len(book_source_genres))

BOOK_AUTHORS: 1488070
BOOK_GENRES: 4981671
BOOK_SOURCE_GENRES: 4291574


## 18. Final Data Preparation Validation

A final validation is performed before the prepared DataFrames are loaded into the SQLite database.

The validation confirms:

- The number of records in each entity and relationship table.
- That all foreign keys were successfully mapped and contain no missing values.
- That primary key IDs are unique within the `BOOKS`, `AUTHORS`, `GENRES`, and `SOURCE_GENRES` tables.

This final checkpoint ensures that the prepared relational data is structurally consistent and ready for database creation and loading.

In [29]:
# =========================================================
# Final Data Preparation Validation
# =========================================================

print("=" * 60)
print("FINAL DATA PREPARATION VALIDATION")
print("=" * 60)

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")

print("\nForeign Key Validation")
print("-" * 60)

print(
    "BOOK_AUTHORS missing book IDs:",
    book_authors["book_id"].isna().sum()
)

print(
    "BOOK_AUTHORS missing author IDs:",
    book_authors["author_id"].isna().sum()
)

print(
    "BOOK_GENRES missing book IDs:",
    book_genres["book_id"].isna().sum()
)

print(
    "BOOK_GENRES missing genre IDs:",
    book_genres["genre_id"].isna().sum()
)

print(
    "BOOK_SOURCE_GENRES missing book IDs:",
    book_source_genres["book_id"].isna().sum()
)

print(
    "BOOK_SOURCE_GENRES missing source genre IDs:",
    book_source_genres["source_genre_id"].isna().sum()
)

print("\nDuplicate Primary Keys")
print("-" * 60)

print("Duplicate book IDs:",
      books["book_id"].duplicated().sum())

print("Duplicate author IDs:",
      authors["author_id"].duplicated().sum())

print("Duplicate genre IDs:",
      genres["genre_id"].duplicated().sum())

print("Duplicate source genre IDs:",
      source_genres["source_genre_id"].duplicated().sum())

FINAL DATA PREPARATION VALIDATION
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574

Foreign Key Validation
------------------------------------------------------------
BOOK_AUTHORS missing book IDs: 0
BOOK_AUTHORS missing author IDs: 0
BOOK_GENRES missing book IDs: 0
BOOK_GENRES missing genre IDs: 0
BOOK_SOURCE_GENRES missing book IDs: 0
BOOK_SOURCE_GENRES missing source genre IDs: 0

Duplicate Primary Keys
------------------------------------------------------------
Duplicate book IDs: 0
Duplicate author IDs: 0
Duplicate genre IDs: 0
Duplicate source genre IDs: 0


## 19. Save Prepared Relational Data

The validated relational DataFrames are saved as individual Parquet files in the `Data/prepared` directory.

Saving each table separately preserves the relational structure and creates reusable, efficient input files for the SQLite database creation process.

These prepared datasets will be used to populate the database tables in the next stage of the project.

In [ ]:
prepared_folder = Path("../Data/prepared")
prepared_folder.mkdir(parents=True, exist_ok=True)

books.to_parquet(
    prepared_folder / "books.parquet",
    index=False
)

authors.to_parquet(
    prepared_folder / "authors.parquet",
    index=False
)

book_authors.to_parquet(
    prepared_folder / "book_authors.parquet",
    index=False
)

genres.to_parquet(
    prepared_folder / "genres.parquet",
    index=False
)

book_genres.to_parquet(
    prepared_folder / "book_genres.parquet",
    index=False
)

source_genres.to_parquet(
    prepared_folder / "source_genres.parquet",
    index=False
)

book_source_genres.to_parquet(
    prepared_folder / "book_source_genres.parquet",
    index=False
)

print("Prepared datasets saved successfully.")

Prepared datasets saved successfully.
